# operations — SLO checks, retention, erasure, maintenance (Snowflake)

One notebook for the operational jobs in tasks/monitor, tasks/privacy and tasks/admin. `op` picks the job.
> generated by: LakeLogic Medallion Builder
> revision: chv1:a5264ea0ab53dafe06f0059818462cccb7f8c7f4df07d6b7e974488159c5b1df
> governance seal: sha256:4eb207b04b65320c4d32ffba41eac7bae8a56f63d105b18e6df1f2a3b308db9c
> source: estate snapshot snap-429b586fd4c04d3d

In [ ]:
# Parameters. A task passes them as one key=value string:
#   EXECUTE NOTEBOOK <DB>.PIPELINE.<NB>('key=value key=value')
# For an interactive run, edit the defaults below.
import sys
# ALL of sys.argv: Snowflake fills it from index 0 (no notebook name in front), so
# skipping index 0 drops the first argument. key=value parsing does not care where each sits.
_kv = dict(t.split('=', 1) for t in ' '.join(sys.argv).split() if '=' in t)
DOMAIN = _kv.get('domain', '')  # the domain whose _system.yaml to load
SYSTEM = _kv.get('system', '')  # the system within that domain
DB = _kv.get('db', '')  # the Snowflake database (the task passes it)
ENVIRONMENT = _kv.get('environment', 'dev')  # a key of _system.yaml environments
OP = _kv.get('op', 'slo')  # slo | retention | maintenance | erasure
FAIL_ON_BREACH = _kv.get('fail_on_breach', 'true')  # fail the task on a breach (slo, retention)
PIPELINE_RUN_ID = _kv.get('pipeline_run_id', '')  # optional link to the run that produced the data
print({'domain': DOMAIN, 'system': SYSTEM, 'db': DB, 'environment': ENVIRONMENT, 'op': OP, 'fail_on_breach': FAIL_ON_BREACH, 'pipeline_run_id': PIPELINE_RUN_ID})

In [ ]:
# lakelogic==1.79.4 from PyPI (deployment target); if this account's notebooks cannot reach PyPI,
# from wheels staged in <DB>.PIPELINE.WHEELS (see tasks/admin/_setup.sql).
import os, subprocess, sys
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *a]).returncode
if _pip('lakelogic==1.79.4', 'polars', 'pandas', 'pyarrow') != 0:
    from snowflake.snowpark.context import get_active_session
    _s = get_active_session(); os.makedirs('/tmp/wl', exist_ok=True)
    _s.file.get(f'@{DB}.PIPELINE.WHEELS', '/tmp/wl')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                    '--find-links', '/tmp/wl', 'lakelogic==1.79.4'], check=True)

In [ ]:
# The registry paths are relative to the repository root; a task's CWD is not guaranteed to be it.
import os, sys
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, '_contracts')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)
SYS = f'_contracts/{DOMAIN}/{SYSTEM}/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB
from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection
session.sql(f'USE DATABASE {DB}').collect()
session.sql(f'USE SCHEMA {DB}.PIPELINE').collect()
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)

In [ ]:
from runtime import operations as ops
if OP == 'slo':
    result = ops.slo_checks(conn, DB, SYS, environment=ENVIRONMENT,
                            fail_on_breach=ops._truthy(FAIL_ON_BREACH), pipeline_run_id=PIPELINE_RUN_ID or None)
elif OP == 'retention':
    result = ops.retention_check(conn, DB, SYS, environment=ENVIRONMENT, fail_on_breach=ops._truthy(FAIL_ON_BREACH))
elif OP == 'maintenance':
    result = ops.maintenance(conn, DB, SYS, environment=ENVIRONMENT)
elif OP == 'erasure':
    # Requests are ROWS in PIPELINE.ERASURE_REQUESTS (tasks/privacy/_erasure.sql); a task
    # cannot take run-time arguments. Dry run unless the row says dry_run=false, confirm=ERASE
    # and carries a case_ref.
    result = []
    for req in session.sql(f"SELECT * FROM {DB}.PIPELINE.ERASURE_REQUESTS WHERE STATUS = 'pending'").collect():
        result.append(ops.erasure(conn, DB, SYS, subject_column=req['SUBJECT_COLUMN'],
            subject_ids=[s.strip() for s in str(req['SUBJECT_IDS']).split(',') if s.strip()],
            environment=ENVIRONMENT, strategy=req['STRATEGY'] or 'nullify',
            dry_run=ops._truthy(req['DRY_RUN']), confirm=req['CONFIRM'] or '', case_ref=req['CASE_REF'] or ''))
else:
    raise ValueError(f'unknown op {OP!r}: slo | retention | maintenance | erasure')
result